In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset
import torchvision.utils as vutils
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F

import matplotlib.pyplot as plt
import ellipse

In [ ]:
# This line sets the device to either a gpu or cpu
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
class ConvolutionalAutoencoder(nn.Module):
    def __init__(self, num_channels, out_size):
        super(ConvolutionalAutoencoder, self).__init__()
        self.econv1 = nn.Conv2d(num_channels, 32, kernel_size = 5, bias = False)
        self.ebatch1 = nn.BatchNorm2d(32)
        self.econv2 = nn.Conv2d(32, 8, kernel_size = 5, bias = False)
        self.ebatch2 = nn.BatchNorm2d(8)
        self.econv3 = nn.Conv2d(8, out_size, kernel_size = 56, bias = True)

        self.dconv1 = nn.ConvTranspose2d(out_size, 8, kernel_size = 56, bias = True)
        self.dbatch1 = nn.BatchNorm2d(8)
        self.dconv2 = nn.ConvTranspose2d(8, 32, kernel_size = 5, bias = False)
        self.dbatch2 = nn.BatchNorm2d(32)
        self.dconv3 = nn.ConvTranspose2d(32, num_channels, kernel_size = 5, bias = False)

    def encoder(self, x):
        y1 = F.elu(self.ebatch1(self.econv1(x)))
        y2 = F.elu(self.ebatch2(self.econv2(y1)))
        z = self.econv3(y2)
        return z

    def decoder(self, z):
        y1 = F.elu(self.dbatch1(self.dconv1(z)))
        y2 = F.elu(self.dbatch2(self.dconv2(y1)))
        x = torch.sigmoid(self.dconv3(y2))
        return x
     
    def forward(self, x):
        z = self.encoder(x)
        y = self.decoder(z)
        return y

In [ ]:
def train(autoencoder, data_loader, num_epochs = 20, learn_rate = 0.5, weight_decay = 0.0):
    optimizer = torch.optim.SGD(autoencoder.parameters(), lr = learn_rate, weight_decay = weight_decay)
    mse_loss = nn.MSELoss()

    for epoch in range(num_epochs):
        tot_loss = 0
        for i, (x, label) in enumerate(data_loader):
            x = x.to(device)
            y = autoencoder(x)

            loss = mse_loss(x, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            tot_loss = tot_loss + loss.data.cpu().numpy()

        print(epoch, ": Loss =", tot_loss)
        
    return autoencoder

In [ ]:
N = 1000           # training data size
batch_size = 100   # set a batch size that works for you (might just use N)
num_channels = 1   # set this to 1 for ellipse (grayscale) and 3 for teapot (colbor)

# Replace the line below with your generated ellipse images training dataset
X = torch.zeros((N, num_channels, 64, 64))

# These are dummy labels needed to create pytorch TensorDataset
y = torch.zeros(N)

train_data = TensorDataset(X, y)
train_loader = DataLoader(train_data, batch_size = batch_size, shuffle = True)

In [ ]:
latent_dim =  # set your latent dimension here

convAE16 = ConvolutionalAutoencoder(num_channels, latent_dim)
convAE16 = train(convAE16.to(device), train_loader)

In [ ]:
# Load the teapot images
# NOTE: there are 10,000 images -- you can subsample a smaller number,
# e.g., 1000 for training and 100 (separate) for testing
X = torch.load("teapot.pth", weights_only = True)

In [ ]:
# Sanity check to make sure tensor has correct size, should be 10,000 x 3 x 64 x 64
print(X.shape)

# Display the first image to make sure it loaded correctly
plt.imshow(X[0,:,:,:].permute((1,2,0)))
plt.show()